### Importing Dataset

In [2]:
import pandas as pd
df = pd.read_csv('D:\\Vidhi\\DM Project\\WhatsappChatAnalyzer\\dataset\\twitter emotions.csv')

In [4]:
df.head()

,id,text,emotion,intensity
0,10857,@ZubairSabirPTI pls dont insult the word 'Molna',anger,0.479
1,10858,@ArcticFantasy I would have almost took offens...,anger,0.458
2,10859,@IllinoisLoyalty that Rutgers game was an abom...,anger,0.562
3,10860,@CozanGaming that's what lisa asked before she...,anger,0.5
4,10861,Sometimes I get mad over something so minuscul...,anger,0.708


### PREPROCESSING

In [5]:
df['text'].isnull().sum()

np.int64(0)

In [ ]:
df['emotion'].value_counts()   # somewhat balance dataset

emotion
fear       3357
anger      2545
joy        2409
sadness    2280
Name: count, dtype: int64

In [8]:
print(df['text'].sample(15).values)

<ArrowStringArray>
[                                                  'in my dream....They were trying to steal my kidney!!!  #blackmarket #whydidiwatchthat',
                                                                   'The outrage over Kessel's tweet by butthurt USA players is hilarious.',
                                                                                                   '@Hayles_101 The three R's depress me.',
                       '@CallofDuty how do u guys determine teams? Cause I'm 80% on shitty teams when I play and I'm fuckin over it #cod ',
        'Some #people already talks about #Halloween \nYou'll get some #dark #music to go with it\n#gothic #bloody #HalloweenHorrorNights',
                  'at least 5 ppl have left my stores in a huff in the past year but ive never had anyone call guest service on me yet so',
                                'Texans and Astros both shut out tonight. Houston, we're back to normal. #texans #Astros #sadness #losers',
 

In [ ]:
import re
from nltk.stem import PorterStemmer
stemmer = PorterStemmer()

def preprocess_text(text):
    text = str(text).lower()
    text = re.sub(r"http\S+|www\S+", "", text)      # remove urls
    text = re.sub(r"@\w+", "", text)                # remove mentions like @ArcticFantasy
    text = re.sub(r"[^a-zA-Z\s]", " ", text)        # remove symbols
    text = re.sub(r"\s+", " ", text).strip()        # remove extra whitespace
    text = re.sub(r"\b\w\b", "", text)              # remove single characters
    text = re.sub(r"\d+", "", text)                 # remove numbers
    text = text.encode('ascii', 'ignore').decode('ascii') #removing emohis                                               #remove emojis

    #Stemming
    text = ' '.join(stemmer.stem(word) for word in text.split())

    return text

In [9]:
preprocess_text('This is a sample text! Visit http://example.com for more info. 😊 #NLP @user123')

'thi is sampl text visit for more info nlp'

In [10]:
df['text'] = df['text'].apply(preprocess_text)

In [11]:
df.head()

,id,text,emotion,intensity
0,10857,pl dont insult the word molna,anger,0.479
1,10858,would have almost took offens to thi if actual...,anger,0.458
2,10859,that rutger game wa an abomin an affront to go...,anger,0.562
3,10860,that what lisa ask befor she start rage at me ...,anger,0.5
4,10861,sometim get mad over someth so minuscul tri to...,anger,0.708


In [ ]:
# Label encoding for emotion labels
from sklearn.preprocessing import LabelEncoder
label_encoder = LabelEncoder()
df['emotion'] = label_encoder.fit_transform(df['emotion'])

In [23]:
df['emotion'].value_counts()

emotion
1    3357
0    2545
2    2409
3    2280
Name: count, dtype: int64

In [24]:
import nltk
from sklearn.feature_extraction.text import TfidfVectorizer
nltk.download('stopwords')

vectorizer = TfidfVectorizer(stop_words='english',max_features=50000,
    ngram_range=(1,2),
    min_df=2,
    max_df=0.95
)
X_tfidf = vectorizer.fit_transform(df['text'])

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\DELL\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [25]:
# test train split
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X_tfidf, df['emotion'], test_size=0.2, random_state=42, stratify=df['emotion'])

## MODEL 

### Logistic Regression

In [17]:
from sklearn.linear_model import LogisticRegression
lr_model = LogisticRegression(max_iter=1000)
lr_model.fit(X_train, y_train)
y_pred = lr_model.predict(X_test)
from sklearn.metrics import classification_report
print(classification_report(y_test, y_pred))


              precision    recall  f1-score   support

       anger       0.94      0.93      0.93       509
        fear       0.91      0.95      0.93       672
         joy       0.97      0.96      0.96       482
     sadness       0.89      0.86      0.87       456

    accuracy                           0.93      2119
   macro avg       0.93      0.92      0.93      2119
weighted avg       0.93      0.93      0.93      2119



In [18]:

from sklearn.metrics import confusion_matrix
confusion_matrix(y_test, y_pred)


array([[472,  14,   5,  18],
       [  5, 639,   2,  26],
       [  5,  12, 461,   4],
       [ 20,  37,   8, 391]])

### 2. SVM 

In [19]:
from sklearn.svm import LinearSVC
svm_model = LinearSVC()
svm_model.fit(X_train, y_train)
y_pred_svm = svm_model.predict(X_test)
print(classification_report(y_test, y_pred_svm))


              precision    recall  f1-score   support

       anger       0.95      0.95      0.95       509
        fear       0.94      0.95      0.94       672
         joy       0.98      0.97      0.98       482
     sadness       0.89      0.88      0.88       456

    accuracy                           0.94      2119
   macro avg       0.94      0.94      0.94      2119
weighted avg       0.94      0.94      0.94      2119



### 3. XGBOOST 

In [26]:
from xgboost import XGBClassifier

xgmodel = XGBClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=5,
    min_child_weight=3,
    subsample=0.8,
    colsample_bytree=0.8,
    gamma=0.1,
    objective='binary:logistic',
    eval_metric='logloss'
)

xgmodel.fit(X_train, y_train)

,"objective objective: typing.Union[str, xgboost.sklearn._SklObjWProto, typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]], NoneType]Specify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'multi:softprob'
,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,0.8
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes fr

In [28]:
print(classification_report(y_test, xgmodel.predict(X_test)))

              precision    recall  f1-score   support

           0       0.97      0.78      0.86       509
           1       0.74      0.94      0.83       672
           2       0.95      0.84      0.89       482
           3       0.88      0.82      0.84       456

    accuracy                           0.85      2119
   macro avg       0.88      0.85      0.86      2119
weighted avg       0.87      0.85      0.86      2119



In [ ]:
# Hyperparameter tuning of XGBoost using GridSearchCV
from sklearn.model_selection import GridSearchCV
param_grid = {
    'n_estimators': [100, 200],
    'learning_rate': [0.01, 0.1],
    'max_depth': [3, 6],
    'subsample': [0.8, 1.0],
    'colsample_bytree': [0.8, 1.0]
}
grid_search = GridSearchCV(estimator=XGBClassifier(objective='binary:logistic', eval_metric='logloss'), param_grid=param_grid, cv=3, n_jobs=-1, verbose=2)
grid_search.fit(X_train, y_train)
print("Best parameters found: ", grid_search.best_params_)
best_xgmodel = grid_search.best_estimator_

### CHAMPION MODEL - INTO PRODUCTION

In [31]:

import joblib
joblib.dump(svm_model, 'D:\\Vidhi\\DM Project\\WhatsappChatAnalyzer\\Trained_Models\\emotion_model.pkl')

['D:\\Vidhi\\DM Project\\WhatsappChatAnalyzer\\Trained_Models\\emotion_model.pkl']

In [32]:
joblib.dump(vectorizer, 'D:\\Vidhi\\DM Project\\WhatsappChatAnalyzer\\Trained_Models\\emotion_vectorizer.pkl')

['D:\\Vidhi\\DM Project\\WhatsappChatAnalyzer\\Trained_Models\\emotion_vectorizer.pkl']